In [13]:
# CELL 1 — Environment Setup (Updated for Colab 2025)
import os

# Fix: update package list first, then install with fallback flag
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless --fix-missing -qq

# Upgrade to PySpark 4.0 to match Colab's environment
!pip install pyspark==4.0.0 --quiet

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Verify Java is actually there
java_check = os.popen("java -version 2>&1").read()
print(f"☕ Java: {java_check}")
print("✅ Environment ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
☕ Java: openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)

✅ Environment ready.


In [14]:
# CELL 2 — SparkSession
from pyspark.sql import SparkSession
import os

# Updated for Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = SparkSession.builder \
    .appName("LocalPulseAI") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.host", "localhost") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark version: {spark.version}")
print("✅ SparkSession is live — LocalPulse AI engine started.")

✅ Spark version: 4.0.0
✅ SparkSession is live — LocalPulse AI engine started.


In [15]:
# CELL 3 — Smoke Test
test_data = [('review_1', 'Great tacos, loved the place!'),
             ('review_2', 'Terrible service, never coming back.'),
             ('review_3', 'Average food, nothing special.')]
columns = ['review_id', 'review_text']
df_test = spark.createDataFrame(test_data, columns)
df_test.show(truncate=False)
print(f'✅ Smoke test passed — {df_test.count()} rows processed by Spark.')

+---------+------------------------------------+
|review_id|review_text                         |
+---------+------------------------------------+
|review_1 |Great tacos, loved the place!       |
|review_2 |Terrible service, never coming back.|
|review_3 |Average food, nothing special.      |
+---------+------------------------------------+

✅ Smoke test passed — 3 rows processed by Spark.


In [16]:
# CELL 4 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify your file is visible to Colab
import os

file_path = "/content/drive/MyDrive/LocalPulseAI/yelp_academic_dataset_review.json"

if os.path.exists(file_path):
    size = os.path.getsize(file_path) / (1024**3)
    print(f"✅ Dataset found — {size:.2f} GB")
else:
    print("❌ File not found — check your Drive path")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset found — 4.98 GB


In [17]:
# CELL 5 — Raw Data Ingestion

print("⏳ Loading Yelp reviews into Spark...")

df_raw = spark.read.json(file_path)

print(f"\n📊 Total reviews loaded: {df_raw.count():,}")
print(f"📋 Total columns: {len(df_raw.columns)}")
print(f"\n🔍 Schema:")
df_raw.printSchema()
print(f"\n👀 Sample rows:")
df_raw.show(5, truncate=50)

⏳ Loading Yelp reviews into Spark...

📊 Total reviews loaded: 6,990,280
📋 Total columns: 9

🔍 Schema:
root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)


👀 Sample rows:
+----------------------+----+-------------------+-----+----------------------+-----+--------------------------------------------------+------+----------------------+
|           business_id|cool|               date|funny|             review_id|stars|                                              text|useful|               user_id|
+----------------------+----+-------------------+-----+----------------------+-----+--------------------------------------------------+------+----------------------+
|XQfwVwDr-v0ZS3_CbbE5Xw|   0|2018-07-07 2

In [18]:
# CELL 6 — Text Preprocessing Pipeline: Lowercasing, Punctuation Removal, Null Filtering

from pyspark.sql.functions import col, lower, regexp_replace, length

print("⏳ Starting text preprocessing pipeline...")

# Step 1: Keep only columns we need
df_selected = df_raw.select("review_id", "business_id", "stars", "text")

# Step 2: Drop nulls in critical columns
df_clean = df_selected.dropna(subset=["text", "stars"])

# Step 3: Lowercase + remove punctuation/numbers/special chars
df_clean = df_clean.withColumn(
    "text_clean",
    regexp_replace(lower(col("text")), r"[^a-z\s]", "")
)

# Step 4: Remove reviews too short to be meaningful
df_clean = df_clean.filter(length(col("text_clean")) > 20)

print(f"📊 Reviews after cleaning: {df_clean.count():,}")
print(f"\n👀 Sample cleaned text:")
df_clean.select("stars", "text", "text_clean").show(3, truncate=60)

⏳ Starting text preprocessing pipeline...
📊 Reviews after cleaning: 6,989,408

👀 Sample cleaned text:
+-----+------------------------------------------------------------+------------------------------------------------------------+
|stars|                                                        text|                                                  text_clean|
+-----+------------------------------------------------------------+------------------------------------------------------------+
|  3.0|If you decide to eat here, just be aware it is going to t...|if you decide to eat here just be aware it is going to ta...|
|  5.0|I've taken a lot of spin classes over the years, and noth...|ive taken a lot of spin classes over the years and nothin...|
|  3.0|Family diner. Had the buffet. Eclectic assortment: a larg...|family diner had the buffet eclectic assortment a large c...|
+-----+------------------------------------------------------------+--------------------------------------------------

In [19]:
# CELL 7 — Tokenization + Stop-word Removal: Splitting text into word tokens, removing noise words

from pyspark.ml.feature import Tokenizer, StopWordsRemover

# Step 1: Tokenization: split sentence into list of words
tokenizer = Tokenizer(inputCol="text_clean", outputCol="words_raw")
df_tokenized = tokenizer.transform(df_clean)

# Step 2: Stop-word removal — drop words like "the", "a", "is", "and"
remover = StopWordsRemover(inputCol="words_raw", outputCol="words_filtered")
df_processed = remover.transform(df_tokenized)

# Checkpoint — see what tokenization looks like
print("✅ Tokenization + Stop-word removal complete!")
print(f"📊 Rows: {df_processed.count():,}")
print("\n👀 Sample — before and after stop-word removal:")
df_processed.select("words_raw", "words_filtered").show(3, truncate=60)

✅ Tokenization + Stop-word removal complete!
📊 Rows: 6,989,408

👀 Sample — before and after stop-word removal:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                   words_raw|                                              words_filtered|
+------------------------------------------------------------+------------------------------------------------------------+
|[if, you, decide, to, eat, here, just, be, aware, it, is,...|[decide, eat, aware, going, take, , hours, beginning, end...|
|[ive, taken, a, lot, of, spin, classes, over, the, years,...|[ive, taken, lot, spin, classes, years, nothing, compares...|
|[family, diner, had, the, buffet, eclectic, assortment, a...|[family, diner, buffet, eclectic, assortment, large, chic...|
+------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 ro

In [20]:
# CELL 8: TF-IDF + Labels (Optimized)
from pyspark.ml.feature import HashingTF, IDF, StringIndexer
from pyspark.sql.functions import when, col

# Step 1: Add sentiment label
df_labeled = df_processed.withColumn(
    "sentiment",
    when(col("stars") <= 2, "negative")
    .when(col("stars") == 3, "neutral")
    .otherwise("positive")
)

# Cache stops Spark recomputing everything upstream repeatedly
df_labeled.cache()
df_labeled.count()  # triggers the cache
print("✅ Labels created and cached")

# Step 2: StringIndexer
indexer = StringIndexer(inputCol="sentiment", outputCol="label")
df_labeled = indexer.fit(df_labeled).transform(df_labeled)

# Step 3: TF
hashingTF = HashingTF(inputCol="words_filtered", outputCol="raw_features", numFeatures=10000)
df_tf = hashingTF.transform(df_labeled)

# Step 4: IDF
idf = IDF(inputCol="raw_features", outputCol="features")
idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)

# Cache final result too
df_tfidf.cache()

print("✅ TF-IDF pipeline complete!")
print(f"📊 Final dataset rows: {df_tfidf.count():,}")
print("\n👀 Label distribution:")
df_tfidf.groupBy("sentiment", "label").count().orderBy("label").show()

✅ Labels created and cached
✅ TF-IDF pipeline complete!
📊 Final dataset rows: 6,989,408

👀 Label distribution:
+---------+-----+-------+
|sentiment|label|  count|
+---------+-----+-------+
| positive|  0.0|4683961|
| negative|  1.0|1613624|
|  neutral|  2.0| 691823|
+---------+-----+-------+



In [21]:
# CELL 9: Save Processed Data: Handoff cell

output_path = "/content/drive/MyDrive/LocalPulseAI/processed/tfidf_features"

print("⏳ Saving processed dataset to Drive...")

df_tfidf.select("review_id", "business_id", "stars", "sentiment", "label", "features") \
    .write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"✅ Data saved to: {output_path}")

⏳ Saving processed dataset to Drive...
✅ Data saved to: /content/drive/MyDrive/LocalPulseAI/processed/tfidf_features
